<a href="https://colab.research.google.com/github/prasansree/BusBookingApp/blob/main/EquipmentStatus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from io import StringIO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────
# 1. RAW DATA  (paste your CSV rows here)
# ──────────────────────────────────────────────
RAW = """SHUTTLEID,LOCATIONID,TEMPERATUREC,MOTORSTATUS,ALARMSTATUS,BATTERYLEVEL,BATTERYHEALTH,LOADWEIGHTKG,LOADTYPE,SHUTTLESPEED,SHUTTLEDIRECTION,MOTORTEMPERATUREC,DOORSTATUS,EMERGENCYSTOPACTIVATED,LASTMAINTENANCEDATE,LASTPOSITIONUPDATE,COMMUNICATIONSTATUS,COMMUNICATIONSIGNALSTRENGTH,FIRMWAREVERSION,OPERATINGMODE,HOURSSINCELASTMAINTENANCE,TOTALOPERATIONALHOURS,TEMPERATUREWARNINGLEVEL,SHUTTLESTATUS,ERRORCODE,ERRORDESCRIPTION,RECORD_CREATED_AT,RECORD_SOURCE
SHUTTLE-001,LOC-A-01,22.90,Running,None,77,95,123.00,Pallet,1.60,Forward,68.00,Closed,FALSE,02/03/2026,2026-02-28 14:17:46.182,Online,80,v1.2.1,Automatic,606,2686,65,Moving,,,2026-02-28 12:15:25.585,kafka_connector
SHUTTLE-006,LOC-C-07,19.20,Running,None,95,95,56.00,Pallet,0.50,Stopped,48.50,Closed,FALSE,02/02/2026,2026-02-28 14:53:51.322,Online,47,v1.2.1,Manual,627,2575,60,Loading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-005,LOC-C-02,21.10,Stopped,None,58,96,0.00,None,0.00,Stopped,37.60,Closed,FALSE,02/04/2026,2026-02-28 14:55:56.328,Online,67,v1.1.8,Automatic,582,2369,65,Idle,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-002,LOC-A-04,25.00,Running,None,96,97,246.00,Pallet,0.60,Forward,51.80,Closed,FALSE,02/03/2026,2026-02-28 14:48:01.334,Online,62,v1.1.8,Manual,610,1680,60,Moving,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-001,LOC-A-01,21.40,Stopped,None,62,94,0.00,None,0.00,Stopped,61.00,Closed,FALSE,02/04/2026,2026-02-28 15:04:06.339,Online,91,v1.1.8,Automatic,572,3124,65,Idle,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-004,LOC-B-05,21.20,Stopped,None,74,96,0.00,None,0.00,Stopped,70.50,Closed,FALSE,02/02/2026,2026-02-28 15:13:11.345,Online,59,v1.1.9,Automatic,627,2177,65,Idle,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-006,LOC-C-07,25.00,Running,Warning,42,96,231.00,Pallet,0.40,Stopped,53.50,Closed,FALSE,02/04/2026,2026-02-28 15:04:16.351,Online,45,v1.1.9,Manual,586,2117,60,Loading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-001,LOC-A-01,22.50,Running,None,64,94,82.00,Pallet,0.70,Stopped,42.40,Closed,FALSE,02/04/2026,2026-02-28 14:15:21.358,Online,78,v1.2.3,Automatic,585,3058,65,Unloading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-003,LOC-B-03,25.80,Running,None,86,98,113.00,Pallet,0.80,Forward,70.50,Closed,FALSE,02/01/2026,2026-02-28 15:07:26.363,Online,58,v1.2.3,Automatic,649,1126,60,Moving,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-004,LOC-B-05,20.10,Running,None,98,93,177.00,Pallet,0.20,Stopped,58.00,Closed,FALSE,02/05/2026,2026-02-28 14:34:31.369,Online,41,v1.1.9,Automatic,566,3844,65,Unloading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-005,LOC-C-02,24.30,Running,None,86,93,147.00,Pallet,0.50,Stopped,57.30,Closed,FALSE,02/01/2026,2026-02-28 14:16:36.374,Online,76,v1.2.1,Automatic,642,3687,65,Loading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-002,LOC-A-04,19.80,Running,Warning,48,94,75.00,Pallet,0.30,Stopped,51.90,Closed,FALSE,02/05/2026,2026-02-28 14:23:41.380,Online,71,v1.2.1,Manual,561,3332,60,Loading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-003,LOC-B-03,21.10,Running,None,87,93,169.00,Pallet,0.40,Stopped,47.20,Closed,FALSE,02/04/2026,2026-02-28 14:26:46.385,Online,47,v1.2.4,Automatic,576,3776,60,Unloading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-008,LOC-D-07,23.40,Running,Warning,41,93,130.00,Pallet,0.30,Stopped,67.70,Closed,FALSE,02/04/2026,2026-02-28 14:56:51.391,Online,53,v1.2.4,Manual,572,3603,60,Unloading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-008,LOC-D-07,21.70,Stopped,None,58,96,0.00,None,0.00,Stopped,49.00,Closed,FALSE,02/02/2026,2026-02-28 14:53:56.397,Online,56,v1.1.9,Manual,616,2085,60,Idle,,,2026-02-28 12:17:25.806,kafka_connector
SHUTTLE-005,LOC-C-02,22.90,Stopped,None,54,98,0.00,None,0.00,Stopped,69.70,Closed,FALSE,02/04/2026,2026-02-28 14:49:01.405,Online,83,v1.1.8,Automatic,589,1393,65,Idle,,,2026-02-28 12:17:25.806,kafka_connector
SHUTTLE-006,LOC-C-07,21.50,Running,None,57,96,168.00,Pallet,0.20,Stopped,67.30,Closed,FALSE,02/02/2026,2026-02-28 14:37:06.411,Online,42,v1.2.3,Manual,633,2370,60,Loading,,,2026-02-28 12:17:25.806,kafka_connector
SHUTTLE-008,LOC-D-07,22.30,Stopped,None,72,97,0.00,None,0.00,Stopped,65.90,Closed,FALSE,02/03/2026,2026-02-28 15:08:11.418,Online,70,v1.2.1,Manual,597,1861,60,Idle,,,2026-02-28 12:17:25.806,kafka_connector
SHUTTLE-007,LOC-D-02,25.40,Stopped,Critical,27,94,0.00,None,0.00,Stopped,60.90,Open,TRUE,02/02/2026,2026-03-08 16:42:28.907,Offline,38,v1.2.3,Automatic,817,3240,60,Error,E301,Battery Critical,2026-03-08 14:39:25.333,kafka_connector
SHUTTLE-001,LOC-A-01,24.70,Fault,Critical,39,92,0.00,None,0.00,Stopped,63.20,Open,TRUE,02/03/2026,2026-03-08 17:21:34.034,Online,56,v1.2.4,Automatic,790,4011,65,Error,E215,Communication Failure,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-002,LOC-A-04,24.70,Stopped,None,88,95,0.00,None,0.00,Stopped,64.70,Closed,FALSE,02/03/2026,2026-03-08 17:08:39.039,Online,74,v1.1.9,Manual,788,2999,60,Idle,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-003,LOC-B-03,24.20,Running,None,91,92,237.00,Pallet,1.50,Reverse,67.50,Closed,FALSE,02/02/2026,2026-03-08 16:53:44.045,Online,63,v1.2.4,Automatic,833,4143,60,Moving,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-003,LOC-B-03,24.90,Stopped,None,72,95,0.00,None,0.00,Stopped,58.10,Closed,FALSE,02/02/2026,2026-03-08 17:20:49.050,Online,65,v1.2.3,Automatic,827,2832,60,Idle,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-001,LOC-A-01,26.00,Stopped,None,64,98,0.00,None,0.00,Stopped,58.60,Closed,FALSE,02/01/2026,2026-03-08 17:31:54.055,Online,84,v1.2.4,Automatic,840,1060,65,Idle,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-001,LOC-A-01,21.40,Running,None,55,97,60.00,Pallet,0.30,Stopped,40.60,Closed,FALSE,02/03/2026,2026-03-08 16:48:59.059,Online,45,v1.2.1,Automatic,797,1654,65,Loading,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-002,LOC-A-04,20.20,Running,None,80,93,154.00,Pallet,0.90,Forward,64.20,Closed,FALSE,02/03/2026,2026-03-08 16:48:04.065,Online,88,v1.2.4,Manual,807,3607,60,Moving,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-007,LOC-D-02,25.90,Running,None,56,93,107.00,Pallet,1.40,Reverse,47.80,Closed,FALSE,02/01/2026,2026-03-08 16:50:09.070,Online,78,v1.1.8,Automatic,842,3572,60,Moving,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-006,LOC-C-07,23.70,Running,None,77,95,109.00,Pallet,1.70,Forward,73.10,Closed,FALSE,02/03/2026,2026-03-08 16:42:14.074,Online,64,v1.2.1,Manual,806,2502,60,Moving,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-007,LOC-D-02,23.50,Stopped,None,84,93,0.00,None,0.00,Stopped,67.90,Closed,FALSE,02/04/2026,2026-03-08 16:59:19.078,Online,55,v1.2.3,Automatic,765,3731,60,Idle,,,2026-03-08 14:40:25.424,kafka_connector"""


# ──────────────────────────────────────────────
# 2. LOAD & FEATURE ENGINEERING
# ──────────────────────────────────────────────
df = pd.read_csv(StringIO(RAW))

# Target: 1 if shuttle is in Error/Fault state
df["LABEL_FAILURE"] = df["SHUTTLESTATUS"].isin(["Error"]).astype(int)

# Encode AlarmStatus: None=0, Warning=1, Critical=2
alarm_map = {"None": 0, "Warning": 1, "Critical": 2}
df["ALARM_ENCODED"] = df["ALARMSTATUS"].map(alarm_map).fillna(0)

# Encode EmergencyStop
df["EMERGENCY_STOP"] = df["EMERGENCYSTOPACTIVATED"].map({"TRUE": 1, "FALSE": 0, True: 1, False: 0}).fillna(0)

# Encode CommunicationStatus
df["COMM_OFFLINE"] = (df["COMMUNICATIONSTATUS"] == "Offline").astype(int)

# Door open flag
df["DOOR_OPEN"] = (df["DOORSTATUS"] == "Open").astype(int)

# Motor fault flag
df["MOTOR_FAULT"] = (df["MOTORSTATUS"] == "Fault").astype(int)

FEATURES = [
    "BATTERYLEVEL",
    "BATTERYHEALTH",
    "MOTORTEMPERATUREC",
    "TEMPERATUREC",
    "HOURSSINCELASTMAINTENANCE",
    "TOTALOPERATIONALHOURS",
    "ALARM_ENCODED",
    "EMERGENCY_STOP",
    "COMM_OFFLINE",
    "DOOR_OPEN",
    "MOTOR_FAULT",
    "COMMUNICATIONSIGNALSTRENGTH",
    "LOADWEIGHTKG",
    "SHUTTLESPEED",
]

X = df[FEATURES]
y = df["LABEL_FAILURE"]

print("=" * 60)
print("  SHUTTLE FAILURE / FAULT PREDICTION")
print("=" * 60)
print(f"\nDataset shape  : {X.shape}")
print(f"Failure cases  : {y.sum()} / {len(y)}")

# ──────────────────────────────────────────────
# 3. TRAIN / TEST SPLIT
# ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ──────────────────────────────────────────────
# 4. MODEL: RANDOM FOREST CLASSIFIER
# ──────────────────────────────────────────────
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=1,
    class_weight="balanced",   # handles class imbalance
    random_state=42,
)
model.fit(X_train, y_train)

# ──────────────────────────────────────────────
# 5. EVALUATION
# ──────────────────────────────────────────────
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

cv_scores = cross_val_score(model, X, y, cv=3, scoring="f1_weighted")

print("\n── Classification Report ──────────────────────")
print(classification_report(y_test, y_pred, target_names=["Normal", "Failure"]))

print("── Confusion Matrix ──────────────────────────")
cm = confusion_matrix(y_test, y_pred)
print(f"              Predicted Normal  Predicted Failure")
print(f"Actual Normal       {cm[0][0]:<18} {cm[0][1]}")
print(f"Actual Failure      {cm[1][0]:<18} {cm[1][1]}")

print(f"\n── Cross-Val F1 (3-fold): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# ──────────────────────────────────────────────
# 6. FEATURE IMPORTANCE
# ──────────────────────────────────────────────
importance_df = (
    pd.DataFrame({"Feature": FEATURES, "Importance": model.feature_importances_})
    .sort_values("Importance", ascending=False)
)
print("\n── Top Feature Importances ──────────────────────")
for _, row in importance_df.iterrows():
    bar = "█" * int(row["Importance"] * 40)
    print(f"  {row['Feature']:<35} {row['Importance']:.4f}  {bar}")

# ──────────────────────────────────────────────
# 7. PREDICT ON ALL SHUTTLES (latest record each)
# ──────────────────────────────────────────────
df_latest = df.sort_values("RECORD_CREATED_AT").groupby("SHUTTLEID").last().reset_index()
X_latest  = df_latest[FEATURES]

df_latest["FAILURE_PROBABILITY"] = model.predict_proba(X_latest)[:, 1]
df_latest["FAILURE_RISK"] = pd.cut(
    df_latest["FAILURE_PROBABILITY"],
    bins=[0, 0.3, 0.6, 1.0],
    labels=["🟢 Low", "🟡 Medium", "🔴 High"],
).astype(object) # Convert to object to allow assigning 'N/A'
df_latest["FAILURE_RISK"] = df_latest["FAILURE_RISK"].fillna("N/A") # Fill None/NaN with 'N/A'
df_latest["ALARMSTATUS"] = df_latest["ALARMSTATUS"].fillna('N/A')

print("\n── Per-Shuttle Failure Risk (latest telemetry) ────")
print(f"{'Shuttle':<14} {'Location':<12} {'Battery%':<10} {'AlarmStatus':<12} "
      f"{'Fail Prob':<11} {'Risk Level'}")
print("-" * 75)
for _, row in df_latest.sort_values("FAILURE_PROBABILITY", ascending=False).iterrows():
    print(
        f"  {row['SHUTTLEID']:<12} {row['LOCATIONID']:<12} {row['BATTERYLEVEL']:<10.0f}"
        f"{row['ALARMSTATUS']:<12} {row['FAILURE_PROBABILITY']:<11.3f} {row['FAILURE_RISK']}"
    )

print("\n✅  Failure prediction complete.\n")


  SHUTTLE FAILURE / FAULT PREDICTION

Dataset shape  : (29, 14)
Failure cases  : 2 / 29

── Classification Report ──────────────────────
              precision    recall  f1-score   support

      Normal       0.88      1.00      0.93         7
     Failure       0.00      0.00      0.00         1

    accuracy                           0.88         8
   macro avg       0.44      0.50      0.47         8
weighted avg       0.77      0.88      0.82         8

── Confusion Matrix ──────────────────────────
              Predicted Normal  Predicted Failure
Actual Normal       7                  0
Actual Failure      1                  0

── Cross-Val F1 (3-fold): 0.902 ± 0.069

── Top Feature Importances ──────────────────────
  DOOR_OPEN                           0.1870  ███████
  BATTERYHEALTH                       0.1797  ███████
  EMERGENCY_STOP                      0.1726  ██████
  TOTALOPERATIONALHOURS               0.1328  █████
  MOTOR_FAULT                         0.1290  █████


In [ ]:
"""
=============================================================
 SHUTTLE MAINTENANCE SCHEDULING PREDICTION
=============================================================
 Predicts the URGENCY SCORE (0-100) for scheduling the
 next maintenance window per shuttle, using a Gradient
 Boosting Regressor.

 The urgency score combines:
   - HoursSinceLastMaintenance  (higher = more urgent)
   - TotalOperationalHours      (wear proxy)
   - BatteryHealth              (lower = more urgent)
   - MotorTemperatureC          (higher = more stress)
   - AlarmStatus                (Warning/Critical = urgent)
   - Error history              (any error code = urgent)
   - ShuttleSpeed & LoadWeightKg (workload intensity)

 Output per shuttle:
   - Urgency score  (0–100)
   - Recommended maintenance window (days from today)
   - Priority tier: 🔴 Critical / 🟡 Soon / 🟢 Routine
=============================================================
"""

import pandas as pd
import numpy as np
from io import StringIO
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1. RAW DATA
# ─────────────────────────────────────────────
RAW = """SHUTTLEID,LOCATIONID,TEMPERATUREC,MOTORSTATUS,ALARMSTATUS,BATTERYLEVEL,BATTERYHEALTH,LOADWEIGHTKG,LOADTYPE,SHUTTLESPEED,SHUTTLEDIRECTION,MOTORTEMPERATUREC,DOORSTATUS,EMERGENCYSTOPACTIVATED,LASTMAINTENANCEDATE,LASTPOSITIONUPDATE,COMMUNICATIONSTATUS,COMMUNICATIONSIGNALSTRENGTH,FIRMWAREVERSION,OPERATINGMODE,HOURSSINCELASTMAINTENANCE,TOTALOPERATIONALHOURS,TEMPERATUREWARNINGLEVEL,SHUTTLESTATUS,ERRORCODE,ERRORDESCRIPTION,RECORD_CREATED_AT,RECORD_SOURCE
SHUTTLE-001,LOC-A-01,22.90,Running,None,77,95,123.00,Pallet,1.60,Forward,68.00,Closed,FALSE,02/03/2026,2026-02-28 14:17:46.182,Online,80,v1.2.1,Automatic,606,2686,65,Moving,,,2026-02-28 12:15:25.585,kafka_connector
SHUTTLE-006,LOC-C-07,19.20,Running,None,95,95,56.00,Pallet,0.50,Stopped,48.50,Closed,FALSE,02/02/2026,2026-02-28 14:53:51.322,Online,47,v1.2.1,Manual,627,2575,60,Loading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-005,LOC-C-02,21.10,Stopped,None,58,96,0.00,None,0.00,Stopped,37.60,Closed,FALSE,02/04/2026,2026-02-28 14:55:56.328,Online,67,v1.1.8,Automatic,582,2369,65,Idle,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-002,LOC-A-04,25.00,Running,None,96,97,246.00,Pallet,0.60,Forward,51.80,Closed,FALSE,02/03/2026,2026-02-28 14:48:01.334,Online,62,v1.1.8,Manual,610,1680,60,Moving,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-001,LOC-A-01,21.40,Stopped,None,62,94,0.00,None,0.00,Stopped,61.00,Closed,FALSE,02/04/2026,2026-02-28 15:04:06.339,Online,91,v1.1.8,Automatic,572,3124,65,Idle,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-004,LOC-B-05,21.20,Stopped,None,74,96,0.00,None,0.00,Stopped,70.50,Closed,FALSE,02/02/2026,2026-02-28 15:13:11.345,Online,59,v1.1.9,Automatic,627,2177,65,Idle,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-006,LOC-C-07,25.00,Running,Warning,42,96,231.00,Pallet,0.40,Stopped,53.50,Closed,FALSE,02/04/2026,2026-02-28 15:04:16.351,Online,45,v1.1.9,Manual,586,2117,60,Loading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-001,LOC-A-01,22.50,Running,None,64,94,82.00,Pallet,0.70,Stopped,42.40,Closed,FALSE,02/04/2026,2026-02-28 14:15:21.358,Online,78,v1.2.3,Automatic,585,3058,65,Unloading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-003,LOC-B-03,25.80,Running,None,86,98,113.00,Pallet,0.80,Forward,70.50,Closed,FALSE,02/01/2026,2026-02-28 15:07:26.363,Online,58,v1.2.3,Automatic,649,1126,60,Moving,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-004,LOC-B-05,20.10,Running,None,98,93,177.00,Pallet,0.20,Stopped,58.00,Closed,FALSE,02/05/2026,2026-02-28 14:34:31.369,Online,41,v1.1.9,Automatic,566,3844,65,Unloading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-005,LOC-C-02,24.30,Running,None,86,93,147.00,Pallet,0.50,Stopped,57.30,Closed,FALSE,02/01/2026,2026-02-28 14:16:36.374,Online,76,v1.2.1,Automatic,642,3687,65,Loading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-002,LOC-A-04,19.80,Running,Warning,48,94,75.00,Pallet,0.30,Stopped,51.90,Closed,FALSE,02/05/2026,2026-02-28 14:23:41.380,Online,71,v1.2.1,Manual,561,3332,60,Loading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-003,LOC-B-03,21.10,Running,None,87,93,169.00,Pallet,0.40,Stopped,47.20,Closed,FALSE,02/04/2026,2026-02-28 14:26:46.385,Online,47,v1.2.4,Automatic,576,3776,60,Unloading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-008,LOC-D-07,23.40,Running,Warning,41,93,130.00,Pallet,0.30,Stopped,67.70,Closed,FALSE,02/04/2026,2026-02-28 14:56:51.391,Online,53,v1.2.4,Manual,572,3603,60,Unloading,,,2026-02-28 12:16:25.908,kafka_connector
SHUTTLE-008,LOC-D-07,21.70,Stopped,None,58,96,0.00,None,0.00,Stopped,49.00,Closed,FALSE,02/02/2026,2026-02-28 14:53:56.397,Online,56,v1.1.9,Manual,616,2085,60,Idle,,,2026-02-28 12:17:25.806,kafka_connector
SHUTTLE-005,LOC-C-02,22.90,Stopped,None,54,98,0.00,None,0.00,Stopped,69.70,Closed,FALSE,02/04/2026,2026-02-28 14:49:01.405,Online,83,v1.1.8,Automatic,589,1393,65,Idle,,,2026-02-28 12:17:25.806,kafka_connector
SHUTTLE-006,LOC-C-07,21.50,Running,None,57,96,168.00,Pallet,0.20,Stopped,67.30,Closed,FALSE,02/02/2026,2026-02-28 14:37:06.411,Online,42,v1.2.3,Manual,633,2370,60,Loading,,,2026-02-28 12:17:25.806,kafka_connector
SHUTTLE-008,LOC-D-07,22.30,Stopped,None,72,97,0.00,None,0.00,Stopped,65.90,Closed,FALSE,02/03/2026,2026-02-28 15:08:11.418,Online,70,v1.2.1,Manual,597,1861,60,Idle,,,2026-02-28 12:17:25.806,kafka_connector
SHUTTLE-007,LOC-D-02,25.40,Stopped,Critical,27,94,0.00,None,0.00,Stopped,60.90,Open,TRUE,02/02/2026,2026-03-08 16:42:28.907,Offline,38,v1.2.3,Automatic,817,3240,60,Error,E301,Battery Critical,2026-03-08 14:39:25.333,kafka_connector
SHUTTLE-001,LOC-A-01,24.70,Fault,Critical,39,92,0.00,None,0.00,Stopped,63.20,Open,TRUE,02/03/2026,2026-03-08 17:21:34.034,Online,56,v1.2.4,Automatic,790,4011,65,Error,E215,Communication Failure,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-002,LOC-A-04,24.70,Stopped,None,88,95,0.00,None,0.00,Stopped,64.70,Closed,FALSE,02/03/2026,2026-03-08 17:08:39.039,Online,74,v1.1.9,Manual,788,2999,60,Idle,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-003,LOC-B-03,24.20,Running,None,91,92,237.00,Pallet,1.50,Reverse,67.50,Closed,FALSE,02/02/2026,2026-03-08 16:53:44.045,Online,63,v1.2.4,Automatic,833,4143,60,Moving,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-003,LOC-B-03,24.90,Stopped,None,72,95,0.00,None,0.00,Stopped,58.10,Closed,FALSE,02/02/2026,2026-03-08 17:20:49.050,Online,65,v1.2.3,Automatic,827,2832,60,Idle,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-001,LOC-A-01,26.00,Stopped,None,64,98,0.00,None,0.00,Stopped,58.60,Closed,FALSE,02/01/2026,2026-03-08 17:31:54.055,Online,84,v1.2.4,Automatic,840,1060,65,Idle,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-001,LOC-A-01,21.40,Running,None,55,97,60.00,Pallet,0.30,Stopped,40.60,Closed,FALSE,02/03/2026,2026-03-08 16:48:59.059,Online,45,v1.2.1,Automatic,797,1654,65,Loading,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-002,LOC-A-04,20.20,Running,None,80,93,154.00,Pallet,0.90,Forward,64.20,Closed,FALSE,02/03/2026,2026-03-08 16:48:04.065,Online,88,v1.2.4,Manual,807,3607,60,Moving,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-007,LOC-D-02,25.90,Running,None,56,93,107.00,Pallet,1.40,Reverse,47.80,Closed,FALSE,02/01/2026,2026-03-08 16:50:09.070,Online,78,v1.1.8,Automatic,842,3572,60,Moving,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-006,LOC-C-07,23.70,Running,None,77,95,109.00,Pallet,1.70,Forward,73.10,Closed,FALSE,02/03/2026,2026-03-08 16:42:14.074,Online,64,v1.2.1,Manual,806,2502,60,Moving,,,2026-03-08 14:40:25.424,kafka_connector
SHUTTLE-007,LOC-D-02,23.50,Stopped,None,84,93,0.00,None,0.00,Stopped,67.90,Closed,FALSE,02/04/2026,2026-03-08 16:59:19.078,Online,55,v1.2.3,Automatic,765,3731,60,Idle,,,2026-03-08 14:40:25.424,kafka_connector"""


# ─────────────────────────────────────────────
# 2. LOAD & FEATURE ENGINEERING
# ─────────────────────────────────────────────
df = pd.read_csv(StringIO(RAW))

# Encode AlarmStatus
alarm_map  = {"None": 0, "Warning": 1, "Critical": 2}
df["ALARM_ENCODED"] = df["ALARMSTATUS"].map(alarm_map).fillna(0)

# Emergency stop
df["EMERGENCY_STOP"] = df["EMERGENCYSTOPACTIVATED"].map(
    {"TRUE": 1, "FALSE": 0, True: 1, False: 0}
).fillna(0)

# Has active error code
df["HAS_ERROR"] = df["ERRORCODE"].notna().astype(int)

# Workload intensity proxy (speed × load)
df["WORKLOAD"] = df["SHUTTLESPEED"] * df["LOADWEIGHTKG"]

# Battery degradation: invert health so low health → high value
df["BATTERY_DEGRADATION"] = 100 - df["BATTERYHEALTH"]

# ─────────────────────────────────────────────
# 3. BUILD ENGINEERED URGENCY LABEL (rule-based)
#    Used as the regression TARGET since we have
#    no historical "time to maintenance" column.
# ─────────────────────────────────────────────
def compute_urgency(row):
    score = 0.0
    # Hours since last maintenance (max expected ~700h → 30 pts)
    score += min(row["HOURSSINCELASTMAINTENANCE"] / 700, 1.0) * 30
    # Total operational hours wear (max 4500h → 20 pts)
    score += min(row["TOTALOPERATIONALHOURS"] / 4500, 1.0) * 20
    # Battery health degradation (0-15 pts)
    score += (row["BATTERY_DEGRADATION"] / 100) * 15
    # Motor temperature (normal <60, concern >70 → 0-15 pts)
    score += min(max(row["MOTORTEMPERATUREC"] - 40, 0) / 40, 1.0) * 15
    # Alarm status (0/1/2 → 0-10 pts)
    score += (row["ALARM_ENCODED"] / 2) * 10
    # Active error (0 or 7 pts)
    score += row["HAS_ERROR"] * 7
    # Emergency stop (0 or 3 pts)
    score += row["EMERGENCY_STOP"] * 3
    return round(min(score, 100), 2)

df["URGENCY_SCORE"] = df.apply(compute_urgency, axis=1)

FEATURES = [
    "HOURSSINCELASTMAINTENANCE",
    "TOTALOPERATIONALHOURS",
    "BATTERY_DEGRADATION",
    "MOTORTEMPERATUREC",
    "TEMPERATUREC",
    "ALARM_ENCODED",
    "HAS_ERROR",
    "EMERGENCY_STOP",
    "WORKLOAD",
    "COMMUNICATIONSIGNALSTRENGTH",
    "BATTERYLEVEL",
]

X = df[FEATURES]
y = df["URGENCY_SCORE"]

print("=" * 60)
print("  SHUTTLE MAINTENANCE SCHEDULING PREDICTION")
print("=" * 60)
print(f"\nDataset shape    : {X.shape}")
print(f"Urgency range    : {y.min():.1f} – {y.max():.1f}")
print(f"Mean urgency     : {y.mean():.1f}")

# ─────────────────────────────────────────────
# 4. MODEL: GRADIENT BOOSTING REGRESSOR
# ─────────────────────────────────────────────
model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_samples_leaf=1,
    subsample=0.8,
    random_state=42,
)
model.fit(X, y)

cv_r2 = cross_val_score(model, X, y, cv=3, scoring="r2")
cv_mae = -cross_val_score(model, X, y, cv=3, scoring="neg_mean_absolute_error")

print(f"\n── Cross-Val R²  (3-fold): {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
print(f"── Cross-Val MAE (3-fold): {cv_mae.mean():.2f} ± {cv_mae.std():.2f}")

# ─────────────────────────────────────────────
# 5. FEATURE IMPORTANCE
# ─────────────────────────────────────────────
importance_df = (
    pd.DataFrame({"Feature": FEATURES, "Importance": model.feature_importances_})
    .sort_values("Importance", ascending=False)
)
print("\n── Top Feature Importances ────────────────────────────")
for _, row in importance_df.iterrows():
    bar = "█" * int(row["Importance"] * 40)
    print(f"  {row['Feature']:<35} {row['Importance']:.4f}  {bar}")

# ─────────────────────────────────────────────
# 6. MAINTENANCE SCHEDULE PER SHUTTLE
# ─────────────────────────────────────────────
df_latest = df.sort_values("RECORD_CREATED_AT").groupby("SHUTTLEID").last().reset_index()
X_latest  = df_latest[FEATURES]

df_latest["PREDICTED_URGENCY"] = model.predict(X_latest).clip(0, 100)

def days_until_maintenance(urgency_score):
    """Higher urgency → fewer days until maintenance."""
    if urgency_score >= 80:
        return 0          # Immediate
    elif urgency_score >= 65:
        return int(7 - (urgency_score - 65) / 15 * 7)
    elif urgency_score >= 45:
        return int(30 - (urgency_score - 45) / 20 * 23)
    else:
        return int(60 - urgency_score * 0.5)

df_latest["DAYS_TO_MAINTENANCE"] = df_latest["PREDICTED_URGENCY"].apply(days_until_maintenance)

def priority_label(score):
    if score >= 80:  return "🔴 CRITICAL – Immediate"
    if score >= 65:  return "🟠 HIGH     – Within 1 week"
    if score >= 45:  return "🟡 MEDIUM   – Within 1 month"
    return               "🟢 ROUTINE  – Within 2 months"

df_latest["PRIORITY"] = df_latest["PREDICTED_URGENCY"].apply(priority_label)

print("\n── Maintenance Schedule (latest telemetry per shuttle) ─")
print(f"{'Shuttle':<14} {'Location':<12} {'Hrs Since Maint':<17} "
      f"{'Urgency':<10} {'Days Until Maint':<18} {'Priority'}")
print("-" * 95)
for _, row in df_latest.sort_values("PREDICTED_URGENCY", ascending=False).iterrows():
    print(
        f"  {row['SHUTTLEID']:<12} {row['LOCATIONID']:<12} "
        f"{row['HOURSSINCELASTMAINTENANCE']:<17.0f} "
        f"{row['PREDICTED_URGENCY']:<10.1f} "
        f"{row['DAYS_TO_MAINTENANCE']:<18} "
        f"{row['PRIORITY']}"
    )

# ─────────────────────────────────────────────
# 7. EXPORT SCHEDULE TO CSV
# ─────────────────────────────────────────────
output_cols = [
    "SHUTTLEID", "LOCATIONID", "HOURSSINCELASTMAINTENANCE",
    "TOTALOPERATIONALHOURS", "BATTERYHEALTH", "ALARMSTATUS",
    "PREDICTED_URGENCY", "DAYS_TO_MAINTENANCE", "PRIORITY"
]
schedule_df = df_latest[output_cols].sort_values("PREDICTED_URGENCY", ascending=False)
schedule_df.to_csv("maintenance_schedule.csv", index=False)
print("\n📁  Maintenance schedule saved to: maintenance_schedule.csv")
print("\n✅  Maintenance scheduling prediction complete.\n")

  SHUTTLE MAINTENANCE SCHEDULING PREDICTION

Dataset shape    : (29, 11)
Urgency range    : 36.1 – 77.7
Mean urgency     : 49.4

── Cross-Val R²  (3-fold): 0.235 ± 0.434
── Cross-Val MAE (3-fold): 4.43 ± 1.24

── Top Feature Importances ────────────────────────────
  TOTALOPERATIONALHOURS               0.2122  ████████
  HAS_ERROR                           0.1876  ███████
  EMERGENCY_STOP                      0.1630  ██████
  MOTORTEMPERATUREC                   0.1439  █████
  ALARM_ENCODED                       0.1171  ████
  BATTERYLEVEL                        0.0787  ███
  HOURSSINCELASTMAINTENANCE           0.0341  █
  BATTERY_DEGRADATION                 0.0272  █
  TEMPERATUREC                        0.0214  
  COMMUNICATIONSIGNALSTRENGTH         0.0137  
  WORKLOAD                            0.0011  

── Maintenance Schedule (latest telemetry per shuttle) ─
Shuttle        Location     Hrs Since Maint   Urgency    Days Until Maint   Priority
---------------------------------------

In [ ]:
pip install "snowflake-connector-python[pandas]" scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pyopenssl to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.9 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
  Attempting uninstall: pyOpenSSL
    Found ex

In [ ]:
"""
=============================================================
 SHUTTLE FAILURE / FAULT PREDICTION
 → Reads from  : EQUIPMENT_STATUS            (Snowflake)
 → Writes to   : SHUTTLE_FAILURE_PREDICTIONS (Snowflake)
=============================================================
 Install deps:
   pip install snowflake-connector-python[pandas] scikit-learn pandas
=============================================================
"""

import os
import warnings
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────
# 1. SNOWFLAKE CONNECTION CONFIG
#    Set these as environment variables, or replace the
#    os.getenv() defaults with your literal values.
#
#    export SF_USER="my_user"
#    export SF_PASSWORD="my_password"
#    export SF_ACCOUNT="xy12345.us-east-1"
#    export SF_WAREHOUSE="COMPUTE_WH"
#    export SF_DATABASE="my_database"
#    export SF_SCHEMA="my_schema"
# ─────────────────────────────────────────────────────────────
SF_CONFIG = {
    "user":      os.getenv("SF_USER",      "your_user"),
    "password":  os.getenv("SF_PASSWORD",  "your_password"),
    "account":   os.getenv("SF_ACCOUNT",   "your_account"),
    "warehouse": os.getenv("SF_WAREHOUSE", "COMPUTE_WH"),
    "database":  os.getenv("SF_DATABASE",  "your_database"),
    "schema":    os.getenv("SF_SCHEMA",    "your_schema"),
    "role":      os.getenv("SF_ROLE",      "your_role"),
}

SOURCE_TABLE  = "EQUIPMENT_STATUS"
TARGET_TABLE  = "SHUTTLE_FAILURE_PREDICTIONS"
MODEL_NAME    = "RandomForestClassifier"
MODEL_VERSION = "v1.0"

FEATURES = [
    "BATTERYLEVEL",
    "BATTERYHEALTH",
    "MOTORTEMPERATUREC",
    "TEMPERATUREC",
    "HOURSSINCELASTMAINTENANCE",
    "TOTALOPERATIONALHOURS",
    "ALARM_ENCODED",
    "EMERGENCY_STOP",
    "COMM_OFFLINE",
    "DOOR_OPEN",
    "MOTOR_FAULT",
    "COMMUNICATIONSIGNALSTRENGTH",
    "LOADWEIGHTKG",
    "SHUTTLESPEED",
]


# ─────────────────────────────────────────────────────────────
# 2. LOAD DATA FROM SNOWFLAKE
# ─────────────────────────────────────────────────────────────
def load_data(conn: snowflake.connector.SnowflakeConnection) -> pd.DataFrame:
    query = f"""
        SELECT
            SHUTTLEID,
            LOCATIONID,
            TEMPERATUREC,
            MOTORSTATUS,
            ALARMSTATUS,
            BATTERYLEVEL,
            BATTERYHEALTH,
            LOADWEIGHTKG,
            SHUTTLESPEED,
            MOTORTEMPERATUREC,
            DOORSTATUS,
            EMERGENCYSTOPACTIVATED,
            COMMUNICATIONSTATUS,
            COMMUNICATIONSIGNALSTRENGTH,
            HOURSSINCELASTMAINTENANCE,
            TOTALOPERATIONALHOURS,
            SHUTTLESTATUS,
            LASTPOSITIONUPDATE
        FROM {SOURCE_TABLE}
        ORDER BY LASTPOSITIONUPDATE DESC
    """
    cur = conn.cursor()
    cur.execute(query)
    df = cur.fetch_pandas_all()
    print(f"✅  Loaded {len(df):,} rows from {SOURCE_TABLE}")
    return df


# ─────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Target label: 1 = Error/Fault state, 0 = Normal
    df["LABEL_FAILURE"] = df["SHUTTLESTATUS"].isin(["Error"]).astype(int)

    # AlarmStatus ordinal: None=0, Warning=1, Critical=2
    alarm_map = {"None": 0, "Warning": 1, "Critical": 2}
    df["ALARM_ENCODED"] = df["ALARMSTATUS"].map(alarm_map).fillna(0).astype(int)

    # Emergency stop → binary int
    df["EMERGENCY_STOP"] = (
        df["EMERGENCYSTOPACTIVATED"].astype(str).str.upper()
        .map({"TRUE": 1, "FALSE": 0}).fillna(0).astype(int)
    )

    # Offline communication flag
    df["COMM_OFFLINE"] = (df["COMMUNICATIONSTATUS"] == "Offline").astype(int)

    # Door open flag
    df["DOOR_OPEN"] = (df["DOORSTATUS"] == "Open").astype(int)

    # Motor fault flag
    df["MOTOR_FAULT"] = (df["MOTORSTATUS"] == "Fault").astype(int)

    return df


# ─────────────────────────────────────────────────────────────
# 4. TRAIN MODEL
# ─────────────────────────────────────────────────────────────
def train_model(df: pd.DataFrame) -> RandomForestClassifier:
    X = df[FEATURES]
    y = df["LABEL_FAILURE"]

    print(f"\n── Training  |  Rows: {len(X)}  |  Failure labels: {y.sum()}/{len(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        class_weight="balanced",   # compensates for class imbalance
        random_state=42,
    )
    model.fit(X_train, y_train)

    # ── Evaluation ──────────────────────────────────────────
    y_pred = model.predict(X_test)
    cv_f1  = cross_val_score(model, X, y, cv=3, scoring="f1_weighted")

    print("\n── Classification Report ──────────────────────────────")
    print(classification_report(y_test, y_pred, target_names=["Normal", "Failure"]))
    print(f"── Cross-Val F1 (3-fold): {cv_f1.mean():.3f} ± {cv_f1.std():.3f}")

    # ── Feature importance ───────────────────────────────────
    imp = (
        pd.DataFrame({"Feature": FEATURES, "Importance": model.feature_importances_})
        .sort_values("Importance", ascending=False)
    )
    print("\n── Feature Importances ────────────────────────────────")
    for _, r in imp.iterrows():
        bar = "█" * int(r["Importance"] * 40)
        print(f"  {r['Feature']:<35} {r['Importance']:.4f}  {bar}")

    return model


# ─────────────────────────────────────────────────────────────
# 5. GENERATE PREDICTIONS (most recent record per shuttle)
# ─────────────────────────────────────────────────────────────
def generate_predictions(df: pd.DataFrame, model: RandomForestClassifier) -> pd.DataFrame:
    df_latest = (
        df.sort_values("LASTPOSITIONUPDATE")
        .groupby("SHUTTLEID").last().reset_index()
    )

    proba = model.predict_proba(df_latest[FEATURES])[:, 1]
    pred  = model.predict(df_latest[FEATURES])

    def risk_level(p: float) -> str:
        if p >= 0.6: return "High"
        if p >= 0.3: return "Medium"
        return "Low"

    now = datetime.now(timezone.utc).replace(tzinfo=None)

    result = pd.DataFrame({
        "SHUTTLEID":                    df_latest["SHUTTLEID"],
        "LOCATIONID":                   df_latest["LOCATIONID"],
        "PREDICTION_TIMESTAMP":         now,
        "SOURCE_RECORD_TIME":           pd.to_datetime(df_latest["LASTPOSITIONUPDATE"]),
        "BATTERYLEVEL":                 df_latest["BATTERYLEVEL"].astype(int),
        "BATTERYHEALTH":                df_latest["BATTERYHEALTH"].astype(int),
        "MOTORTEMPERATUREC":            df_latest["MOTORTEMPERATUREC"].astype(float),
        "ALARMSTATUS":                  df_latest["ALARMSTATUS"],
        "EMERGENCYSTOPACTIVATED":       df_latest["EMERGENCYSTOPACTIVATED"].astype(str),
        "COMMUNICATIONSTATUS":          df_latest["COMMUNICATIONSTATUS"],
        "COMMUNICATIONSIGNALSTRENGTH":  df_latest["COMMUNICATIONSIGNALSTRENGTH"].astype(int),
        "DOORSTATUS":                   df_latest["DOORSTATUS"],
        "MOTORSTATUS":                  df_latest["MOTORSTATUS"],
        "HOURSSINCELASTMAINTENANCE":    df_latest["HOURSSINCELASTMAINTENANCE"].astype(int),
        "TOTALOPERATIONALHOURS":        df_latest["TOTALOPERATIONALHOURS"].astype(int),
        "FAILURE_PROBABILITY":          proba.round(4),
        "FAILURE_PREDICTION":           pred.astype(int),
        "RISK_LEVEL":                   [risk_level(p) for p in proba],
        "MODEL_NAME":                   MODEL_NAME,
        "MODEL_VERSION":                MODEL_VERSION,
        "CREATED_AT":                   now,
    })

    # ── Console summary ──────────────────────────────────────
    print("\n── Per-Shuttle Predictions ────────────────────────────")
    print(f"{'Shuttle':<14} {'Location':<12} {'Fail Prob':>10}  {'Risk':<8}  Prediction")
    print("─" * 62)
    for _, r in result.sort_values("FAILURE_PROBABILITY", ascending=False).iterrows():
        label = "⚠️  FAILURE" if r["FAILURE_PREDICTION"] == 1 else "✅  Normal"
        print(
            f"  {r['SHUTTLEID']:<12} {r['LOCATIONID']:<12} "
            f"{r['FAILURE_PROBABILITY']:>10.4f}  {r['RISK_LEVEL']:<8}  {label}"
        )

    return result


# ─────────────────────────────────────────────────────────────
# 6. WRITE RESULTS BACK TO SNOWFLAKE
# ─────────────────────────────────────────────────────────────
def write_to_snowflake(
    conn: snowflake.connector.SnowflakeConnection,
    df_results: pd.DataFrame,
    table: str,
) -> None:
    success, nchunks, nrows, _ = write_pandas(
        conn,
        df_results,
        table_name=table,
        auto_create_table=False,   # table must exist — run DDL script first
        quote_identifiers=False,
    )
    if success:
        print(f"\n✅  {nrows} prediction rows written → {table}  ({nchunks} chunk(s))")
    else:
        raise RuntimeError(f"write_pandas failed for table {table}")


# ─────────────────────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 62)
    print("  SHUTTLE FAILURE / FAULT PREDICTION  (Snowflake)")
    print("=" * 62)

    conn = snowflake.connector.connect(**SF_CONFIG)
    try:
        df_raw     = load_data(conn)
        df         = engineer_features(df_raw)
        model      = train_model(df)
        df_results = generate_predictions(df, model)
        write_to_snowflake(conn, df_results, TARGET_TABLE)
    finally:
        conn.close()
        print("\n🔒  Snowflake connection closed.")


if __name__ == "__main__":
    main()

In [ ]:
"""
=============================================================
 SHUTTLE MAINTENANCE SCHEDULING PREDICTION
 → Reads from  : EQUIPMENT_STATUS                (Snowflake)
 → Writes to   : SHUTTLE_MAINTENANCE_PREDICTIONS (Snowflake)
=============================================================
 Install deps:
   pip install snowflake-connector-python[pandas] scikit-learn pandas
=============================================================
"""

import os
import warnings
from datetime import datetime, timezone, timedelta

import pandas as pd
import numpy as np
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────
# 1. SNOWFLAKE CONNECTION CONFIG
#    Set these as environment variables, or replace the
#    os.getenv() defaults with your literal values.
#
#    export SF_USER="my_user"
#    export SF_PASSWORD="my_password"
#    export SF_ACCOUNT="xy12345.us-east-1"
#    export SF_WAREHOUSE="COMPUTE_WH"
#    export SF_DATABASE="my_database"
#    export SF_SCHEMA="my_schema"
# ─────────────────────────────────────────────────────────────
SF_CONFIG = {
    "user":      os.getenv("SF_USER",      "your_user"),
    "password":  os.getenv("SF_PASSWORD",  "your_password"),
    "account":   os.getenv("SF_ACCOUNT",   "your_account"),
    "warehouse": os.getenv("SF_WAREHOUSE", "COMPUTE_WH"),
    "database":  os.getenv("SF_DATABASE",  "your_database"),
    "schema":    os.getenv("SF_SCHEMA",    "your_schema"),
    "role":      os.getenv("SF_ROLE",      "your_role"),
}

SOURCE_TABLE  = "EQUIPMENT_STATUS"
TARGET_TABLE  = "SHUTTLE_MAINTENANCE_PREDICTIONS"
MODEL_NAME    = "GradientBoostingRegressor"
MODEL_VERSION = "v1.0"

FEATURES = [
    "HOURSSINCELASTMAINTENANCE",
    "TOTALOPERATIONALHOURS",
    "BATTERY_DEGRADATION",
    "MOTORTEMPERATUREC",
    "TEMPERATUREC",
    "ALARM_ENCODED",
    "HAS_ERROR",
    "EMERGENCY_STOP",
    "WORKLOAD",
    "COMMUNICATIONSIGNALSTRENGTH",
    "BATTERYLEVEL",
]


# ─────────────────────────────────────────────────────────────
# 2. LOAD DATA FROM SNOWFLAKE
# ─────────────────────────────────────────────────────────────
def load_data(conn: snowflake.connector.SnowflakeConnection) -> pd.DataFrame:
    query = f"""
        SELECT
            SHUTTLEID,
            LOCATIONID,
            TEMPERATUREC,
            MOTORSTATUS,
            ALARMSTATUS,
            BATTERYLEVEL,
            BATTERYHEALTH,
            LOADWEIGHTKG,
            SHUTTLESPEED,
            MOTORTEMPERATUREC,
            EMERGENCYSTOPACTIVATED,
            COMMUNICATIONSIGNALSTRENGTH,
            HOURSSINCELASTMAINTENANCE,
            TOTALOPERATIONALHOURS,
            SHUTTLESTATUS,
            ERRORCODE,
            ERRORDESCRIPTION,
            LASTPOSITIONUPDATE
        FROM {SOURCE_TABLE}
        ORDER BY LASTPOSITIONUPDATE DESC
    """
    cur = conn.cursor()
    cur.execute(query)
    df = cur.fetch_pandas_all()
    print(f"✅  Loaded {len(df):,} rows from {SOURCE_TABLE}")
    return df


# ─────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING + URGENCY LABEL
# ─────────────────────────────────────────────────────────────
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # AlarmStatus ordinal: None=0, Warning=1, Critical=2
    alarm_map = {"None": 0, "Warning": 1, "Critical": 2}
    df["ALARM_ENCODED"] = df["ALARMSTATUS"].map(alarm_map).fillna(0).astype(int)

    # Emergency stop → binary int
    df["EMERGENCY_STOP"] = (
        df["EMERGENCYSTOPACTIVATED"].astype(str).str.upper()
        .map({"TRUE": 1, "FALSE": 0}).fillna(0).astype(int)
    )

    # Active error code flag
    df["HAS_ERROR"] = df["ERRORCODE"].notna().astype(int)

    # Workload intensity proxy: speed × load weight
    df["WORKLOAD"] = df["SHUTTLESPEED"] * df["LOADWEIGHTKG"]

    # Battery degradation: invert health so lower health → higher value
    df["BATTERY_DEGRADATION"] = (100 - df["BATTERYHEALTH"]).astype(float)

    # ── Rule-based urgency score (regression target) ─────────
    # Used as the supervised label since no historical
    # "time to maintenance" column is available in the source.
    # Decomposed into weighted sub-scores (total = 100):
    #   30 pts  HoursSinceLastMaintenance  (normalised to 700 h)
    #   20 pts  TotalOperationalHours wear (normalised to 4500 h)
    #   15 pts  BatteryHealth degradation
    #   15 pts  MotorTemperature stress    (above 40 °C)
    #   10 pts  AlarmStatus level
    #    7 pts  Active error code
    #    3 pts  Emergency stop triggered
    def urgency(row) -> float:
        score = 0.0
        score += min(row["HOURSSINCELASTMAINTENANCE"] / 700,  1.0) * 30
        score += min(row["TOTALOPERATIONALHOURS"]     / 4500, 1.0) * 20
        score += (row["BATTERY_DEGRADATION"] / 100)                * 15
        score += min(max(row["MOTORTEMPERATUREC"] - 40, 0) / 40, 1.0) * 15
        score += (row["ALARM_ENCODED"] / 2)                        * 10
        score += row["HAS_ERROR"]                                  *  7
        score += row["EMERGENCY_STOP"]                             *  3
        return round(min(score, 100), 2)

    df["URGENCY_SCORE"] = df.apply(urgency, axis=1)
    return df


# ─────────────────────────────────────────────────────────────
# 4. TRAIN MODEL
# ─────────────────────────────────────────────────────────────
def train_model(df: pd.DataFrame) -> GradientBoostingRegressor:
    X = df[FEATURES]
    y = df["URGENCY_SCORE"]

    print(f"\n── Training  |  Rows: {len(X)}  |  Urgency range: {y.min():.1f}–{y.max():.1f}")

    model = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=42,
    )
    model.fit(X, y)

    # ── Evaluation ──────────────────────────────────────────
    cv_r2  = cross_val_score(model, X, y, cv=3, scoring="r2")
    cv_mae = -cross_val_score(model, X, y, cv=3, scoring="neg_mean_absolute_error")
    print(f"── Cross-Val R²  (3-fold): {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
    print(f"── Cross-Val MAE (3-fold): {cv_mae.mean():.2f} ± {cv_mae.std():.2f}")

    # ── Feature importance ───────────────────────────────────
    imp = (
        pd.DataFrame({"Feature": FEATURES, "Importance": model.feature_importances_})
        .sort_values("Importance", ascending=False)
    )
    print("\n── Feature Importances ────────────────────────────────")
    for _, r in imp.iterrows():
        bar = "█" * int(r["Importance"] * 40)
        print(f"  {r['Feature']:<35} {r['Importance']:.4f}  {bar}")

    return model


# ─────────────────────────────────────────────────────────────
# 5. GENERATE PREDICTIONS (most recent record per shuttle)
# ─────────────────────────────────────────────────────────────
def days_until_maintenance(urgency: float) -> int:
    """Map urgency score → recommended days until maintenance."""
    if urgency >= 80:  return 0
    if urgency >= 65:  return int(7  - (urgency - 65) / 15 * 7)
    if urgency >= 45:  return int(30 - (urgency - 45) / 20 * 23)
    return int(60 - urgency * 0.5)


def priority_tier(urgency: float) -> str:
    if urgency >= 80:  return "CRITICAL"
    if urgency >= 65:  return "HIGH"
    if urgency >= 45:  return "MEDIUM"
    return "ROUTINE"


def generate_predictions(df: pd.DataFrame, model: GradientBoostingRegressor) -> pd.DataFrame:
    df_latest = (
        df.sort_values("LASTPOSITIONUPDATE")
        .groupby("SHUTTLEID").last().reset_index()
    )

    urgency_scores = model.predict(df_latest[FEATURES]).clip(0, 100)
    today          = datetime.now(timezone.utc).date()
    now            = datetime.now(timezone.utc).replace(tzinfo=None)

    days_list  = [days_until_maintenance(u) for u in urgency_scores]
    rec_dates  = [today + timedelta(days=d) for d in days_list]
    priorities = [priority_tier(u) for u in urgency_scores]

    result = pd.DataFrame({
        "SHUTTLEID":                  df_latest["SHUTTLEID"],
        "LOCATIONID":                 df_latest["LOCATIONID"],
        "PREDICTION_TIMESTAMP":       now,
        "SOURCE_RECORD_TIME":         pd.to_datetime(df_latest["LASTPOSITIONUPDATE"]),
        "HOURSSINCELASTMAINTENANCE":  df_latest["HOURSSINCELASTMAINTENANCE"].astype(int),
        "TOTALOPERATIONALHOURS":      df_latest["TOTALOPERATIONALHOURS"].astype(int),
        "BATTERYHEALTH":              df_latest["BATTERYHEALTH"].astype(int),
        "MOTORTEMPERATUREC":          df_latest["MOTORTEMPERATUREC"].astype(float),
        "ALARMSTATUS":                df_latest["ALARMSTATUS"],
        "ERRORCODE":                  df_latest["ERRORCODE"].fillna(""),
        "ERRORDESCRIPTION":           df_latest["ERRORDESCRIPTION"].fillna(""),
        "URGENCY_SCORE":              urgency_scores.round(2),
        "DAYS_TO_MAINTENANCE":        days_list,
        "RECOMMENDED_DATE":           rec_dates,
        "PRIORITY_TIER":              priorities,
        "MODEL_NAME":                 MODEL_NAME,
        "MODEL_VERSION":              MODEL_VERSION,
        "CREATED_AT":                 now,
    })

    # ── Console summary ──────────────────────────────────────
    print("\n── Maintenance Schedule ───────────────────────────────")
    print(f"{'Shuttle':<14} {'Location':<12} {'Hrs/Maint':>9}  {'Urgency':>8}  {'Days':>5}  {'Rec. Date':<12}  Priority")
    print("─" * 82)
    for _, r in result.sort_values("URGENCY_SCORE", ascending=False).iterrows():
        emoji = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡", "ROUTINE": "🟢"}.get(r["PRIORITY_TIER"], "")
        print(
            f"  {r['SHUTTLEID']:<12} {r['LOCATIONID']:<12} "
            f"{r['HOURSSINCELASTMAINTENANCE']:>9}  "
            f"{r['URGENCY_SCORE']:>8.1f}  "
            f"{r['DAYS_TO_MAINTENANCE']:>5}  "
            f"{str(r['RECOMMENDED_DATE']):<12}  "
            f"{emoji} {r['PRIORITY_TIER']}"
        )

    return result


# ─────────────────────────────────────────────────────────────
# 6. WRITE RESULTS BACK TO SNOWFLAKE
# ─────────────────────────────────────────────────────────────
def write_to_snowflake(
    conn: snowflake.connector.SnowflakeConnection,
    df_results: pd.DataFrame,
    table: str,
) -> None:
    success, nchunks, nrows, _ = write_pandas(
        conn,
        df_results,
        table_name=table,
        auto_create_table=False,   # table must exist — run DDL script first
        quote_identifiers=False,
    )
    if success:
        print(f"\n✅  {nrows} prediction rows written → {table}  ({nchunks} chunk(s))")
    else:
        raise RuntimeError(f"write_pandas failed for table {table}")


# ─────────────────────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 62)
    print("  SHUTTLE MAINTENANCE SCHEDULING PREDICTION  (Snowflake)")
    print("=" * 62)

    conn = snowflake.connector.connect(**SF_CONFIG)
    try:
        df_raw     = load_data(conn)
        df         = engineer_features(df_raw)
        model      = train_model(df)
        df_results = generate_predictions(df, model)
        write_to_snowflake(conn, df_results, TARGET_TABLE)
    finally:
        conn.close()
        print("\n🔒  Snowflake connection closed.")


if __name__ == "__main__":
    main()

In [2]:
pip install snowflake-connector-python[pandas] scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pyopenssl to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
  Attempting uninstall: pyOpenSSL
    Found 

In [7]:
"""
=============================================================
 SHUTTLE FAILURE / FAULT PREDICTION
 → Reads from  : EQUIPMENT_STATUS            (Snowflake)
 → Writes to   : SHUTTLE_FAILURE_PREDICTIONS (Snowflake)
=============================================================
 Install deps:
   pip install snowflake-connector-python[pandas] scikit-learn pandas
=============================================================
"""

import os
import warnings
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────
# 1. SNOWFLAKE CONNECTION CONFIG
#    Set these as environment variables, or replace the
#    os.getenv() defaults with your literal values.
#
#    export SF_USER="my_user"
#    export SF_PASSWORD="my_password"
#    export SF_ACCOUNT="xy12345.us-east-1"
#    export SF_WAREHOUSE="COMPUTE_WH"
#    export SF_DATABASE="my_database"
#    export SF_SCHEMA="my_schema"
# ─────────────────────────────────────────────────────────────
SF_CONFIG = {
    "user":      os.getenv("SF_USER",      "SREEMANOKARAN"),
    "password":  os.getenv("SF_PASSWORD",  "Pass42day#8769"),  # ⚠️ rotate your password first
    "account":   os.getenv("SF_ACCOUNT",   "DDEJIFN-CXA58599"),
    "warehouse": os.getenv("SF_WAREHOUSE", "COMPUTE_WH"),
    "database":  os.getenv("SF_DATABASE",  "MATERIAL_HANDLING_DB"),
    "schema":    os.getenv("SF_SCHEMA",    "BRONZE_EQUIPMENT_STS"),
}

SOURCE_TABLE  = "BRONZE_EQUIPMENT_STS.EQUIPMENT_STATUS"
TARGET_TABLE  = "BRONZE_EQUIPMENT_STS.SHUTTLE_FAILURE_PREDICTIONS"
MODEL_NAME    = "RandomForestClassifier"
MODEL_VERSION = "v1.0"

FEATURES = [
    "BATTERYLEVEL",
    "BATTERYHEALTH",
    "MOTORTEMPERATUREC",
    "TEMPERATUREC",
    "HOURSSINCELASTMAINTENANCE",
    "TOTALOPERATIONALHOURS",
    "ALARM_ENCODED",
    "EMERGENCY_STOP",
    "COMM_OFFLINE",
    "DOOR_OPEN",
    "MOTOR_FAULT",
    "COMMUNICATIONSIGNALSTRENGTH",
    "LOADWEIGHTKG",
    "SHUTTLESPEED",
]


# ─────────────────────────────────────────────────────────────
# 2. LOAD DATA FROM SNOWFLAKE
# ─────────────────────────────────────────────────────────────
def load_data(conn: snowflake.connector.SnowflakeConnection) -> pd.DataFrame:
    query = f"""
        SELECT
            SHUTTLEID,
            LOCATIONID,
            TEMPERATUREC,
            MOTORSTATUS,
            ALARMSTATUS,
            BATTERYLEVEL,
            BATTERYHEALTH,
            LOADWEIGHTKG,
            SHUTTLESPEED,
            MOTORTEMPERATUREC,
            DOORSTATUS,
            EMERGENCYSTOPACTIVATED,
            COMMUNICATIONSTATUS,
            COMMUNICATIONSIGNALSTRENGTH,
            HOURSSINCELASTMAINTENANCE,
            TOTALOPERATIONALHOURS,
            SHUTTLESTATUS,
            LASTPOSITIONUPDATE
        FROM {SOURCE_TABLE}
        ORDER BY LASTPOSITIONUPDATE DESC
    """
    cur = conn.cursor()
    cur.execute(query)
    df = cur.fetch_pandas_all()
    print(f"✅  Loaded {len(df):,} rows from {SOURCE_TABLE}")
    return df


# ─────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Target label: 1 = Error/Fault state, 0 = Normal
    df["LABEL_FAILURE"] = df["SHUTTLESTATUS"].isin(["Error"]).astype(int)

    # AlarmStatus ordinal: None=0, Warning=1, Critical=2
    alarm_map = {"None": 0, "Warning": 1, "Critical": 2}
    df["ALARM_ENCODED"] = df["ALARMSTATUS"].map(alarm_map).fillna(0).astype(int)

    # Emergency stop → binary int
    df["EMERGENCY_STOP"] = (
        df["EMERGENCYSTOPACTIVATED"].astype(str).str.upper()
        .map({"TRUE": 1, "FALSE": 0}).fillna(0).astype(int)
    )

    # Offline communication flag
    df["COMM_OFFLINE"] = (df["COMMUNICATIONSTATUS"] == "Offline").astype(int)

    # Door open flag
    df["DOOR_OPEN"] = (df["DOORSTATUS"] == "Open").astype(int)

    # Motor fault flag
    df["MOTOR_FAULT"] = (df["MOTORSTATUS"] == "Fault").astype(int)

    return df


# ─────────────────────────────────────────────────────────────
# 4. TRAIN MODEL
# ─────────────────────────────────────────────────────────────
def train_model(df: pd.DataFrame) -> RandomForestClassifier:
    X = df[FEATURES]
    y = df["LABEL_FAILURE"]

    print(f"\n── Training  |  Rows: {len(X)}  |  Failure labels: {y.sum()}/{len(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        class_weight="balanced",   # compensates for class imbalance
        random_state=42,
    )
    model.fit(X_train, y_train)

    # ── Evaluation ──────────────────────────────────────────
    y_pred = model.predict(X_test)
    cv_f1  = cross_val_score(model, X, y, cv=3, scoring="f1_weighted")

    print("\n── Classification Report ──────────────────────────────")
    print(classification_report(y_test, y_pred, target_names=["Normal", "Failure"]))
    print(f"── Cross-Val F1 (3-fold): {cv_f1.mean():.3f} ± {cv_f1.std():.3f}")

    # ── Feature importance ───────────────────────────────────
    imp = (
        pd.DataFrame({"Feature": FEATURES, "Importance": model.feature_importances_})
        .sort_values("Importance", ascending=False)
    )
    print("\n── Feature Importances ────────────────────────────────")
    for _, r in imp.iterrows():
        bar = "█" * int(r["Importance"] * 40)
        print(f"  {r['Feature']:<35} {r['Importance']:.4f}  {bar}")

    return model


# ─────────────────────────────────────────────────────────────
# 5. GENERATE PREDICTIONS (most recent record per shuttle)
# ─────────────────────────────────────────────────────────────
def generate_predictions(df: pd.DataFrame, model: RandomForestClassifier) -> pd.DataFrame:
    df_latest = (
        df.sort_values("LASTPOSITIONUPDATE")
        .groupby("SHUTTLEID").last().reset_index()
    )

    proba = model.predict_proba(df_latest[FEATURES])[:, 1]
    pred  = model.predict(df_latest[FEATURES])

    def risk_level(p: float) -> str:
        if p >= 0.6: return "High"
        if p >= 0.3: return "Medium"
        return "Low"

    now = datetime.now(timezone.utc).replace(tzinfo=None)

    result = pd.DataFrame({
        "SHUTTLEID":                    df_latest["SHUTTLEID"],
        "LOCATIONID":                   df_latest["LOCATIONID"],
        "PREDICTION_TIMESTAMP":         now,
        "SOURCE_RECORD_TIME":           pd.to_datetime(df_latest["LASTPOSITIONUPDATE"]),
        "BATTERYLEVEL":                 df_latest["BATTERYLEVEL"].astype(int),
        "BATTERYHEALTH":                df_latest["BATTERYHEALTH"].astype(int),
        "MOTORTEMPERATUREC":            df_latest["MOTORTEMPERATUREC"].astype(float),
        "ALARMSTATUS":                  df_latest["ALARMSTATUS"],
        "EMERGENCYSTOPACTIVATED":       df_latest["EMERGENCYSTOPACTIVATED"].astype(str),
        "COMMUNICATIONSTATUS":          df_latest["COMMUNICATIONSTATUS"],
        "COMMUNICATIONSIGNALSTRENGTH":  df_latest["COMMUNICATIONSIGNALSTRENGTH"].astype(int),
        "DOORSTATUS":                   df_latest["DOORSTATUS"],
        "MOTORSTATUS":                  df_latest["MOTORSTATUS"],
        "HOURSSINCELASTMAINTENANCE":    df_latest["HOURSSINCELASTMAINTENANCE"].astype(int),
        "TOTALOPERATIONALHOURS":        df_latest["TOTALOPERATIONALHOURS"].astype(int),
        "FAILURE_PROBABILITY":          proba.round(4),
        "FAILURE_PREDICTION":           pred.astype(int),
        "RISK_LEVEL":                   [risk_level(p) for p in proba],
        "MODEL_NAME":                   MODEL_NAME,
        "MODEL_VERSION":                MODEL_VERSION,
        "CREATED_AT":                   now,
    })

    # ── Console summary ──────────────────────────────────────
    print("\n── Per-Shuttle Predictions ────────────────────────────")
    print(f"{'Shuttle':<14} {'Location':<12} {'Fail Prob':>10}  {'Risk':<8}  Prediction")
    print("─" * 62)
    for _, r in result.sort_values("FAILURE_PROBABILITY", ascending=False).iterrows():
        label = "⚠️  FAILURE" if r["FAILURE_PREDICTION"] == 1 else "✅  Normal"
        print(
            f"  {r['SHUTTLEID']:<12} {r['LOCATIONID']:<12} "
            f"{r['FAILURE_PROBABILITY']:>10.4f}  {r['RISK_LEVEL']:<8}  {label}"
        )

    return result


# ─────────────────────────────────────────────────────────────
# 6. WRITE RESULTS BACK TO SNOWFLAKE
# ─────────────────────────────────────────────────────────────
def write_to_snowflake(
    conn: snowflake.connector.SnowflakeConnection,
    df_results: pd.DataFrame,
    table: str,
) -> None:
    success, nchunks, nrows, _ = write_pandas(
        conn,
        df_results,
        table_name=table,
        auto_create_table=False,   # table must exist — run DDL script first
        quote_identifiers=False,
    )
    if success:
        print(f"\n✅  {nrows} prediction rows written → {table}  ({nchunks} chunk(s))")
    else:
        raise RuntimeError(f"write_pandas failed for table {table}")


# ─────────────────────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 62)
    print("  SHUTTLE FAILURE / FAULT PREDICTION  (Snowflake)")
    print("=" * 62)

    conn = snowflake.connector.connect(**SF_CONFIG)
    try:
        df_raw     = load_data(conn)
        df         = engineer_features(df_raw)
        model      = train_model(df)
        df_results = generate_predictions(df, model)
        write_to_snowflake(conn, df_results, TARGET_TABLE)
    finally:
        conn.close()
        print("\n🔒  Snowflake connection closed.")


if __name__ == "__main__":
    main()


  SHUTTLE FAILURE / FAULT PREDICTION  (Snowflake)
✅  Loaded 29 rows from BRONZE_EQUIPMENT_STS.EQUIPMENT_STATUS

── Training  |  Rows: 29  |  Failure labels: 2/29

── Classification Report ──────────────────────────────
              precision    recall  f1-score   support

      Normal       0.88      1.00      0.93         7
     Failure       0.00      0.00      0.00         1

    accuracy                           0.88         8
   macro avg       0.44      0.50      0.47         8
weighted avg       0.77      0.88      0.82         8

── Cross-Val F1 (3-fold): 0.902 ± 0.069

── Feature Importances ────────────────────────────────
  EMERGENCY_STOP                      0.2312  █████████
  DOOR_OPEN                           0.1574  ██████
  BATTERYLEVEL                        0.1559  ██████
  COMMUNICATIONSIGNALSTRENGTH         0.1473  █████
  ALARM_ENCODED                       0.1233  ████
  COMM_OFFLINE                        0.1147  ████
  TEMPERATUREC                        0.0